In [2]:
import pandas as pd


# ============================================================
# Configuration
# ============================================================

input_file = "data.xlsx"
output_file = "data_encoded.xlsx"
mapping_file = "encoding_mappings.xlsx"


# ============================================================
# Load Data
# ============================================================

df = pd.read_excel(input_file, engine="openpyxl")


# ============================================================
# Helper Function
# ============================================================

def create_letter_mapping(series):
    """Create a sequential letter-based mapping for unique values."""
    unique_values = series.dropna().unique()

    return {
        value: (
            "".join(
                chr(65 + ((index - 1) // (26 ** power)) % 26)
                for power in range(
                    int(__import__("math").log(index, 26)), -1, -1
                )
            )
        )
        for index, value in enumerate(unique_values, start=1)
    }


# ============================================================
# Encode National IDs
# ============================================================

# Convert to string to preserve leading zeros.
df["national_id"] = df["national_id"].astype(str)

national_ids = df["national_id"].unique()

national_mapping = {
    value: f"N{i:06d}"
    for i, value in enumerate(national_ids, start=1)
}

df["national_id"] = df["national_id"].map(national_mapping)


# ============================================================
# Encode Shipper Codes
# ============================================================

df["shipper_code"] = df["shipper_code"].astype(str)

shipper_codes = df["shipper_code"].unique()

shipper_mapping = {
    value: f"S{i:06d}"
    for i, value in enumerate(shipper_codes, start=1)
}

df["shipper_code"] = df["shipper_code"].map(shipper_mapping)


# ============================================================
# Encode Province Codes
# ============================================================

province_mapping = create_letter_mapping(df["province_code"])

df["province_code"] = df["province_code"].map(province_mapping)


# ============================================================
# Encode Product Codes
# ============================================================

kala_mapping = create_letter_mapping(df["kala_code"])

df["kala_code"] = df["kala_code"].map(kala_mapping)


# ============================================================
# Preserve b_count
# ============================================================

# The b_count variable is retained without modification.


# ============================================================
# Save Encoded Dataset
# ============================================================

df.to_excel(output_file, index=False)


# ============================================================
# Create Mapping Tables
# ============================================================

national_mapping_df = pd.DataFrame(
    national_mapping.items(),
    columns=["original_national_id", "encoded_national_id"]
)

shipper_mapping_df = pd.DataFrame(
    shipper_mapping.items(),
    columns=["original_shipper_code", "encoded_shipper_code"]
)

province_mapping_df = pd.DataFrame(
    province_mapping.items(),
    columns=["original_province_code", "encoded_province_code"]
)

kala_mapping_df = pd.DataFrame(
    kala_mapping.items(),
    columns=["original_kala_code", "encoded_kala_code"]
)


# ============================================================
# Save Mapping Tables
# ============================================================

with pd.ExcelWriter(mapping_file, engine="openpyxl") as writer:
    national_mapping_df.to_excel(
        writer,
        sheet_name="national_id",
        index=False
    )

    shipper_mapping_df.to_excel(
        writer,
        sheet_name="shipper_code",
        index=False
    )

    province_mapping_df.to_excel(
        writer,
        sheet_name="province_code",
        index=False
    )

    kala_mapping_df.to_excel(
        writer,
        sheet_name="kala_code",
        index=False
    )

In [3]:
import pandas as pd


# ============================================================
# Configuration
# ============================================================

input_file = "data_encoded.xlsx"


# ============================================================
# Load Data
# ============================================================

df = pd.read_excel(input_file, engine="openpyxl")


# ============================================================
# Descriptive Statistics
# ============================================================

total_records = len(df)

unique_national_ids = df["national_id"].nunique(dropna=True)

unique_shipper_codes = df["shipper_code"].nunique(dropna=True)

unique_products = df["kala_code"].nunique(dropna=True)

unique_provinces = df["province_code"].nunique(dropna=True)

unique_product_destination_pairs = (
    df[["kala_code", "province_code"]]
    .drop_duplicates()
    .shape[0]
)


# ============================================================
# Data Quality Checks
# ============================================================

total_missing_values = df.isna().sum().sum()

duplicate_records = df.duplicated().sum()


# ============================================================
# Display Results
# ============================================================

print("=" * 50)
print("Initial Descriptive Statistics")
print("=" * 50)

print(f"Total records:                    {total_records:,}")
print(f"Unique national IDs:             {unique_national_ids:,}")
print(f"Unique shipper codes:            {unique_shipper_codes:,}")
print(f"Unique products:                 {unique_products:,}")
print(f"Unique destination provinces:    {unique_provinces:,}")
print(
    f"Unique product-destination pairs: "
    f"{unique_product_destination_pairs:,}"
)
print(f"Total missing values:            {total_missing_values:,}")
print(f"Duplicate records:               {duplicate_records:,}")

print("=" * 50)

Initial Descriptive Statistics
Total records:                    33,135
Unique national IDs:             3,008
Unique shipper codes:            287
Unique products:                 63
Unique destination provinces:    31
Unique product-destination pairs: 904
Total missing values:            0
Duplicate records:               0


In [4]:
import pandas as pd
import numpy as np


# ============================================================
# Configuration
# ============================================================

input_file = "data_encoded.xlsx"
output_file = "behavioral_features.xlsx"


# ============================================================
# Load Data
# ============================================================

df = pd.read_excel(
    input_file,
    engine="openpyxl"
)

# Ensure that b_count is numeric.
df["b_count"] = pd.to_numeric(
    df["b_count"],
    errors="coerce"
)


# ============================================================
# Helper Functions
# ============================================================

def entropy(series, weights=None):
    """Calculate Shannon entropy."""
    if weights is None:
        counts = series.value_counts()
        probabilities = counts / counts.sum()
    else:
        temp = pd.DataFrame({
            "category": series,
            "weight": weights
        })

        weighted_counts = (
            temp.groupby("category")["weight"]
            .sum()
        )

        probabilities = (
            weighted_counts / weighted_counts.sum()
        )

    probabilities = probabilities[probabilities > 0]

    return -(probabilities * np.log(probabilities)).sum()


def dominant_category_ratio(series, weights):
    """Calculate the weighted share of the dominant category."""
    temp = pd.DataFrame({
        "category": series,
        "weight": weights
    })

    category_load = (
        temp.groupby("category")["weight"]
        .sum()
    )

    total_load = category_load.sum()

    if total_load == 0:
        return 0

    return category_load.max() / total_load


def dominant_category(series, weights):
    """Identify the category with the highest weighted load."""
    temp = pd.DataFrame({
        "category": series,
        "weight": weights
    })

    category_load = (
        temp.groupby("category")["weight"]
        .sum()
        .sort_values(ascending=False)
    )

    if category_load.empty:
        return np.nan

    return category_load.index[0]


# ============================================================
# Extract Behavioral Features
# ============================================================

features = []

for national_id, group in df.groupby("national_id"):

    # Activity volume
    record_count = len(group)

    total_b_count = group["b_count"].sum()

    avg_b_count = (
        total_b_count / record_count
        if record_count > 0
        else 0
    )

    # Product diversity
    kala_diversity = group["kala_code"].nunique()

    dominant_kala = dominant_category(
        group["kala_code"],
        group["b_count"]
    )

    dominant_kala_ratio = dominant_category_ratio(
        group["kala_code"],
        group["b_count"]
    )

    kala_entropy = entropy(
        group["kala_code"],
        group["b_count"]
    )

    # Destination diversity
    destination_diversity = group["province_code"].nunique()

    dominant_destination = dominant_category(
        group["province_code"],
        group["b_count"]
    )

    dominant_destination_ratio = dominant_category_ratio(
        group["province_code"],
        group["b_count"]
    )

    destination_entropy = entropy(
        group["province_code"],
        group["b_count"]
    )

    # Shipper diversity
    shipper_diversity = group["shipper_code"].nunique()

    # Product-destination diversity
    kala_destination_diversity = (
        group[["kala_code", "province_code"]]
        .drop_duplicates()
        .shape[0]
    )

    # Relative diversity measures
    kala_record_ratio = (
        kala_diversity / record_count
        if record_count > 0
        else 0
    )

    destination_record_ratio = (
        destination_diversity / record_count
        if record_count > 0
        else 0
    )

    shipper_record_ratio = (
        shipper_diversity / record_count
        if record_count > 0
        else 0
    )

    kala_destination_record_ratio = (
        kala_destination_diversity / record_count
        if record_count > 0
        else 0
    )

    # Store features for the current national ID
    features.append({
        "national_id": national_id,

        "record_count": record_count,
        "total_b_count": total_b_count,
        "avg_b_count": avg_b_count,

        "kala_diversity": kala_diversity,
        "dominant_kala": dominant_kala,
        "dominant_kala_ratio": dominant_kala_ratio,
        "kala_entropy": kala_entropy,

        "destination_diversity": destination_diversity,
        "dominant_destination": dominant_destination,
        "dominant_destination_ratio": dominant_destination_ratio,
        "destination_entropy": destination_entropy,

        "shipper_diversity": shipper_diversity,

        "kala_destination_diversity":
            kala_destination_diversity,

        "kala_record_ratio": kala_record_ratio,
        "destination_record_ratio":
            destination_record_ratio,
        "shipper_record_ratio":
            shipper_record_ratio,
        "kala_destination_record_ratio":
            kala_destination_record_ratio
    })


# ============================================================
# Create Feature Matrix
# ============================================================

feature_df = pd.DataFrame(features)


# ============================================================
# Sort Feature Matrix
# ============================================================

feature_df = (
    feature_df
    .sort_values("national_id")
    .reset_index(drop=True)
)


# ============================================================
# Quality Checks
# ============================================================

print(f"Number of entities: {feature_df.shape[0]:,}")
print(f"Number of features: {feature_df.shape[1]:,}")

missing_values = feature_df.isna().sum()

if missing_values.sum() > 0:
    print("\nMissing values:")
    print(missing_values[missing_values > 0])


# ============================================================
# Save Feature Matrix
# ============================================================

feature_df.to_excel(
    output_file,
    index=False
)

Number of entities: 3,008
Number of features: 18


In [5]:
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# Configuration
# ============================================================

input_file = "behavioral_features.xlsx"
correlation_output = "feature_correlation_matrix.xlsx"
scatter_output = "behavioral_features_scatter_matrix.png"

correlation_threshold = 0.80


# ============================================================
# Load Behavioral Feature Matrix
# ============================================================

df = pd.read_excel(
    input_file,
    engine="openpyxl"
)


# ============================================================
# Select Candidate Features
# ============================================================

selected_features = [
    "kala_entropy",
    "destination_diversity",
    "dominant_destination_ratio",
    "destination_entropy",
    "shipper_diversity",
    "kala_destination_diversity",
    "kala_record_ratio",
    "destination_record_ratio",
    "shipper_record_ratio",
    "kala_destination_record_ratio"
]

X = df[selected_features].copy()


# ============================================================
# Missing Value Check
# ============================================================

missing_values = X.isna().sum()

if missing_values.sum() > 0:
    print("Missing values:")
    print(missing_values[missing_values > 0])


# ============================================================
# Descriptive Statistics
# ============================================================

descriptive_statistics = X.describe().T

std_values = X.std()

low_variance_features = std_values[
    std_values < 0.01
]


# ============================================================
# Correlation Analysis
# ============================================================

correlation_matrix = X.corr(method="pearson")

correlation_matrix.to_excel(
    correlation_output,
    index=True
)


# ============================================================
# Identify Highly Correlated Feature Pairs
# ============================================================

high_correlations = []

for i in range(len(correlation_matrix.columns)):
    for j in range(i + 1, len(correlation_matrix.columns)):

        feature_1 = correlation_matrix.columns[i]
        feature_2 = correlation_matrix.columns[j]

        correlation = correlation_matrix.iloc[i, j]

        if abs(correlation) >= correlation_threshold:
            high_correlations.append({
                "feature_1": feature_1,
                "feature_2": feature_2,
                "correlation": correlation
            })


high_corr_df = pd.DataFrame(high_correlations)

if not high_corr_df.empty:
    high_corr_df = high_corr_df.sort_values(
        "correlation",
        key=abs,
        ascending=False
    )


# ============================================================
# Prepare Features for Visualization
# ============================================================

plot_df = X.rename(
    columns={
        "kala_entropy": "Kala Entropy",
        "destination_diversity": "Destination Diversity",
        "dominant_destination_ratio": "Dominant Destination Ratio",
        "destination_entropy": "Destination Entropy",
        "shipper_diversity": "Shipper Diversity",
        "kala_destination_diversity": "Kala-Destination Diversity",
        "kala_record_ratio": "Kala/Record Ratio",
        "destination_record_ratio": "Destination/Record Ratio",
        "shipper_record_ratio": "Shipper/Record Ratio",
        "kala_destination_record_ratio": "Kala-Destination/Record Ratio"
    }
)


# ============================================================
# Generate Scatter Matrix
# ============================================================

axes = pd.plotting.scatter_matrix(
    plot_df,
    figsize=(22, 22),
    diagonal="hist",
    alpha=0.7
)

for ax in axes.flatten():
    ax.tick_params(
        axis="both",
        labelsize=7
    )

plt.suptitle(
    "Scatter Matrix of Behavioral Features",
    fontsize=18,
    y=0.90
)

plt.tight_layout()

plt.savefig(
    scatter_output,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler


# ============================================================
# Configuration
# ============================================================

input_file = "behavioral_features.xlsx"

output_standardized = "final_standardized_features.xlsx"
output_statistics = "standardization_statistics.xlsx"


# ============================================================
# Load Behavioral Feature Matrix
# ============================================================

df = pd.read_excel(
    input_file,
    engine="openpyxl"
)


# ============================================================
# Define Final Features
# ============================================================

final_features = [
    "kala_entropy",
    "destination_entropy",
    "shipper_diversity",
    "kala_destination_diversity",
    "kala_record_ratio",
    "destination_record_ratio",
    "shipper_record_ratio"
]


# ============================================================
# Validate Required Columns
# ============================================================

missing_columns = [
    column
    for column in final_features
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing:\n"
        + "\n".join(missing_columns)
    )


# ============================================================
# Create Feature Matrix
# ============================================================

X = df[final_features].copy()
national_ids = df["national_id"].copy()


# ============================================================
# Validate Missing Values
# ============================================================

missing_values = X.isna().sum()

if missing_values.sum() > 0:
    raise ValueError(
        "Missing values were detected in the selected features."
    )


# ============================================================
# Ensure Numeric Data Types
# ============================================================

for column in final_features:
    X[column] = pd.to_numeric(
        X[column],
        errors="coerce"
    )

if X.isna().sum().sum() > 0:
    raise ValueError(
        "Non-numeric values were detected in the selected features."
    )


# ============================================================
# Check Feature Variance
# ============================================================

variance = X.var(ddof=0)

zero_variance = variance[variance == 0]

if not zero_variance.empty:
    raise ValueError(
        "Zero-variance features were detected:\n"
        + "\n".join(zero_variance.index)
    )


# ============================================================
# Statistics Before Standardization
# ============================================================

before_stats = pd.DataFrame({
    "feature": final_features,
    "mean_before": X.mean().values,
    "std_before": X.std(ddof=0).values,
    "min_before": X.min().values,
    "max_before": X.max().values
})


# ============================================================
# Z-Score Standardization
# ============================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


# ============================================================
# Create Standardized Feature Matrix
# ============================================================

X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=final_features
)

X_scaled_df.insert(
    0,
    "national_id",
    national_ids.values
)


# ============================================================
# Validate Standardization
# ============================================================

mean_after = X_scaled_df[final_features].mean()
std_after = X_scaled_df[final_features].std(ddof=0)

after_stats = pd.DataFrame({
    "feature": final_features,
    "mean_after": mean_after.values,
    "std_after": std_after.values,
    "min_after": X_scaled_df[final_features].min().values,
    "max_after": X_scaled_df[final_features].max().values
})

mean_ok = np.allclose(
    mean_after.values,
    0,
    atol=1e-10
)

std_ok = np.allclose(
    std_after.values,
    1,
    atol=1e-10
)

if not (mean_ok and std_ok):
    raise ValueError(
        "Standardization validation failed."
    )


# ============================================================
# Create Standardization Statistics Table
# ============================================================

statistics = before_stats.merge(
    after_stats,
    on="feature"
)


# ============================================================
# Save Standardized Feature Matrix
# ============================================================

X_scaled_df.to_excel(
    output_standardized,
    index=False
)


# ============================================================
# Save Standardization Statistics
# ============================================================

statistics.to_excel(
    output_statistics,
    index=False
)


# ============================================================
# Prepare Model Input Matrix
# ============================================================

# Exclude national_id from the numerical model input.
X_model = X_scaled


# ============================================================
# Final Validation
# ============================================================

print(
    f"Standardization completed successfully. "
    f"Model input shape: {X_model.shape}"
)

Standardization completed successfully. Model input shape: (3008, 7)


In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest


# ============================================================
# Configuration
# ============================================================

input_file = "final_standardized_features.xlsx"
output_file = "isolation_forest_results.xlsx"

N_ESTIMATORS = 500
CONTAMINATION = 0.10
RANDOM_STATE = 42


# ============================================================
# Load Standardized Feature Matrix
# ============================================================

df = pd.read_excel(
    input_file,
    engine="openpyxl"
)


# ============================================================
# Define Model Features
# ============================================================

features = [
    "kala_entropy",
    "destination_entropy",
    "shipper_diversity",
    "kala_destination_diversity",
    "kala_record_ratio",
    "destination_record_ratio",
    "shipper_record_ratio"
]


# ============================================================
# Validate Required Columns
# ============================================================

missing_columns = [
    column
    for column in features
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing:\n"
        + "\n".join(missing_columns)
    )


# ============================================================
# Prepare Model Input
# ============================================================

X = df[features].copy()
national_ids = df["national_id"].copy()


# ============================================================
# Validate Input Data
# ============================================================

if X.isna().sum().sum() > 0:
    raise ValueError(
        "Missing values were detected in the model input matrix."
    )

if not X.apply(
    lambda column: pd.api.types.is_numeric_dtype(column)
).all():
    raise ValueError(
        "All model features must be numeric."
    )


# ============================================================
# Initialize Isolation Forest
# ============================================================

model = IsolationForest(
    n_estimators=N_ESTIMATORS,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1
)


# ============================================================
# Fit Model
# ============================================================

model.fit(X)


# ============================================================
# Calculate Anomaly Scores
# ============================================================

decision_function = model.decision_function(X)

score_samples = model.score_samples(X)

# Lower decision_function values indicate greater anomaly.
# Negating the value makes higher anomaly_score values
# correspond to more anomalous observations.
anomaly_score = -decision_function


# ============================================================
# Generate Model Predictions
# ============================================================

prediction = model.predict(X)

# Isolation Forest output:
#  1  = Normal
# -1  = Anomaly

status = np.where(
    prediction == -1,
    "Anomaly",
    "Normal"
)


# ============================================================
# Create Results Table
# ============================================================

results = pd.DataFrame({
    "national_id": national_ids,
    "anomaly_score": anomaly_score,
    "decision_function": decision_function,
    "score_samples": score_samples,
    "prediction": prediction,
    "status": status
})


# ============================================================
# Rank Observations by Anomaly Score
# ============================================================

results = (
    results
    .sort_values(
        "anomaly_score",
        ascending=False
    )
    .reset_index(drop=True)
)

results["rank"] = np.arange(
    1,
    len(results) + 1
)


# ============================================================
# Organize Result Columns
# ============================================================

results = results[
    [
        "rank",
        "national_id",
        "anomaly_score",
        "decision_function",
        "score_samples",
        "prediction",
        "status"
    ]
]


# ============================================================
# Add Standardized Features to Results
# ============================================================

results_with_features = results.merge(
    df[["national_id"] + features],
    on="national_id",
    how="left"
)


# ============================================================
# Save Results
# ============================================================

results_with_features.to_excel(
    output_file,
    index=False
)


# ============================================================
# Summary
# ============================================================

anomaly_count = (
    results["prediction"] == -1
).sum()

normal_count = (
    results["prediction"] == 1
).sum()

anomaly_percentage = (
    anomaly_count / len(results) * 100
)

print(
    f"Isolation Forest completed successfully. "
    f"Total: {len(results):,}, "
    f"Anomalies: {anomaly_count:,}, "
    f"Normal: {normal_count:,}, "
    f"Anomaly rate: {anomaly_percentage:.2f}%"
)

c:\Users\NITRO\anaconda3\envs\environment\lib\site-packages\sklearn\ensemble\iforest.py:247: FutureWarning: behaviour="old" is deprecated and will be removed in version 0.22. Please use behaviour="new", which makes the decision_function change to match other anomaly detection algorithm API.
  FutureWarning)
c:\Users\NITRO\anaconda3\envs\environment\lib\site-packages\sklearn\utils\fixes.py:223: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if _joblib.__version__ >= LooseVersion('0.12'):
c:\Users\NITRO\anaconda3\envs\environment\lib\site-packages\sklearn\ensemble\iforest.py:415: DeprecationWarning: threshold_ attribute is deprecated in 0.20 and will be removed in 0.22.
  " be removed in 0.22.", DeprecationWarning)


Isolation Forest completed successfully. Total: 3,008, Anomalies: 301, Normal: 2,707, Anomaly rate: 10.01%


In [8]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from itertools import product


# ============================================================
# Configuration
# ============================================================

input_file = "final_standardized_features.xlsx"

tuning_output_file = "isolation_forest_tuning.xlsx"
stability_output_file = "isolation_forest_stability.xlsx"
final_output_file = "isolation_forest_final_results.xlsx"


# ------------------------------------------------------------
# Random States
# ------------------------------------------------------------

# Random state used for the final model and contamination analysis
RANDOM_STATE = 42

# Random states used for stability analysis
RANDOM_STATES = [42, 52, 62, 72, 82]

# Random state used for sensitivity analysis
TUNING_RANDOM_STATE = 42


# ------------------------------------------------------------
# Hyperparameter Search Space
# ------------------------------------------------------------

N_ESTIMATORS_VALUES = [100, 200, 300, 500]

MAX_SAMPLES_VALUES = [0.5, 0.75, 1.0]

MAX_FEATURES_VALUES = [0.5, 0.75, 1.0]


# ------------------------------------------------------------
# Contamination values for separate analysis
# ------------------------------------------------------------

CONTAMINATION_VALUES = [0.01, 0.02, 0.05, 0.10]


# ------------------------------------------------------------
# Top-K percentages
# ------------------------------------------------------------

TOP_K_PERCENTAGES = [0.01, 0.05, 0.10]


# ------------------------------------------------------------
# Final contamination
# ------------------------------------------------------------

FINAL_CONTAMINATION = 0.05


# ============================================================
# Load Standardized Feature Matrix
# ============================================================

df = pd.read_excel(
    input_file,
    engine="openpyxl"
)


# ============================================================
# Define Model Features
# ============================================================

features = [
    "kala_entropy",
    "destination_entropy",
    "shipper_diversity",
    "kala_destination_diversity",
    "kala_record_ratio",
    "destination_record_ratio",
    "shipper_record_ratio"
]


# ============================================================
# Validate Required Columns
# ============================================================

required_columns = features + ["national_id"]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing:\n"
        + "\n".join(missing_columns)
    )


# ============================================================
# Validate national_id
# ============================================================

if df["national_id"].isna().any():
    raise ValueError(
        "Missing values were detected in national_id."
    )


if df["national_id"].duplicated().any():
    duplicated_count = df["national_id"].duplicated().sum()

    raise ValueError(
        f"Duplicate national_id values were detected "
        f"({duplicated_count} duplicated rows). "
        "Each national_id must uniquely identify one observation."
    )


# ============================================================
# Prepare Model Input
# ============================================================

X = df[features].copy()

national_ids = df["national_id"].copy()


# ============================================================
# Validate Input Data
# ============================================================

if X.isna().sum().sum() > 0:
    raise ValueError(
        "Missing values were detected in the model input matrix."
    )


if not X.apply(
    lambda column: pd.api.types.is_numeric_dtype(column)
).all():

    raise ValueError(
        "All model features must be numeric."
    )


# ============================================================
# Utility Function:
# Calculate Anomaly Scores
# ============================================================

def calculate_anomaly_scores(model, X):

    decision_function = model.decision_function(X)

    score_samples = model.score_samples(X)

    # Higher anomaly_score = more anomalous
    anomaly_score = -decision_function

    return (
        anomaly_score,
        decision_function,
        score_samples
    )


# ============================================================
# Utility Function:
# Get Top-K IDs
# ============================================================

def get_top_k_ids(
    anomaly_scores,
    national_ids,
    percentage
):

    n = len(anomaly_scores)

    k = max(
        1,
        int(np.ceil(n * percentage))
    )

    temp = pd.DataFrame({
        "national_id": national_ids.values,
        "anomaly_score": anomaly_scores
    })

    top_k = (
        temp
        .sort_values(
            "anomaly_score",
            ascending=False
        )
        .head(k)
    )

    return set(top_k["national_id"])


# ============================================================
# Utility Function:
# Jaccard Similarity
# ============================================================

def jaccard_similarity(set_a, set_b):

    union = set_a | set_b

    if len(union) == 0:
        return 1.0

    intersection = set_a & set_b

    return len(intersection) / len(union)


# ============================================================
# STEP 1
# Hyperparameter Sensitivity Analysis
# ============================================================

print("\n" + "=" * 70)
print("STEP 1: HYPERPARAMETER SENSITIVITY ANALYSIS")
print("=" * 70)


parameter_combinations = list(
    product(
        N_ESTIMATORS_VALUES,
        MAX_SAMPLES_VALUES,
        MAX_FEATURES_VALUES
    )
)


tuning_results = []


for combination_number, (
    n_estimators,
    max_samples,
    max_features
) in enumerate(
    parameter_combinations,
    start=1
):

    print(
        f"Testing combination "
        f"{combination_number}/{len(parameter_combinations)}: "
        f"n_estimators={n_estimators}, "
        f"max_samples={max_samples}, "
        f"max_features={max_features}"
    )


    # --------------------------------------------------------
    # Isolation Forest
    # --------------------------------------------------------

    model = IsolationForest(
        n_estimators=n_estimators,
        max_samples=max_samples,
        max_features=max_features,
        contamination="auto",
        random_state=TUNING_RANDOM_STATE,
        n_jobs=-1
    )


    model.fit(X)


    # --------------------------------------------------------
    # Calculate anomaly scores
    # --------------------------------------------------------

    anomaly_score, _, _ = calculate_anomaly_scores(
        model,
        X
    )


    # --------------------------------------------------------
    # Score statistics
    # --------------------------------------------------------

    mean_score = np.mean(anomaly_score)

    std_score = np.std(anomaly_score)

    q50 = np.percentile(
        anomaly_score,
        50
    )

    q90 = np.percentile(
        anomaly_score,
        90
    )

    q95 = np.percentile(
        anomaly_score,
        95
    )

    q99 = np.percentile(
        anomaly_score,
        99
    )


    # --------------------------------------------------------
    # Top-K analysis
    # --------------------------------------------------------

    top_k_sets = {}

    for percentage in TOP_K_PERCENTAGES:

        top_k_sets[percentage] = get_top_k_ids(
            anomaly_score,
            national_ids,
            percentage
        )


    # --------------------------------------------------------
    # Top 10% score separation
    # --------------------------------------------------------

    top_10 = top_k_sets[0.10]

    top_10_mask = national_ids.isin(top_10)


    top_10_mean = anomaly_score[
        top_10_mask.values
    ].mean()


    remaining_mean = anomaly_score[
        ~top_10_mask.values
    ].mean()


    score_separation = (
        top_10_mean
        - remaining_mean
    )


    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    tuning_results.append({

        "n_estimators": n_estimators,

        "max_samples": max_samples,

        "max_features": max_features,

        "mean_anomaly_score": mean_score,

        "std_anomaly_score": std_score,

        "median_anomaly_score": q50,

        "q90_anomaly_score": q90,

        "q95_anomaly_score": q95,

        "q99_anomaly_score": q99,

        "top_10_mean_score": top_10_mean,

        "remaining_mean_score": remaining_mean,

        "score_separation": score_separation

    })


tuning_results = pd.DataFrame(
    tuning_results
)


# ============================================================
# STEP 2
# Stability Analysis
# ============================================================

print("\n" + "=" * 70)
print("STEP 2: STABILITY ANALYSIS")
print("=" * 70)


stability_results = []


for combination_number, (
    n_estimators,
    max_samples,
    max_features
) in enumerate(
    parameter_combinations,
    start=1
):

    print(
        f"Stability analysis "
        f"{combination_number}/{len(parameter_combinations)}"
    )


    top_k_by_seed = {
        percentage: []
        for percentage in TOP_K_PERCENTAGES
    }


    scores_by_seed = []


    # --------------------------------------------------------
    # Run model with different random states
    # --------------------------------------------------------

    for random_state in RANDOM_STATES:

        model = IsolationForest(
            n_estimators=n_estimators,
            max_samples=max_samples,
            max_features=max_features,
            contamination="auto",
            random_state=random_state,
            n_jobs=-1
        )


        model.fit(X)


        anomaly_score, _, _ = calculate_anomaly_scores(
            model,
            X
        )


        scores_by_seed.append(
            anomaly_score
        )


        for percentage in TOP_K_PERCENTAGES:

            top_k = get_top_k_ids(
                anomaly_score,
                national_ids,
                percentage
            )

            top_k_by_seed[
                percentage
            ].append(
                top_k
            )


    # --------------------------------------------------------
    # Calculate pairwise Jaccard stability
    # --------------------------------------------------------

    stability_metrics = {}


    for percentage in TOP_K_PERCENTAGES:

        sets = top_k_by_seed[percentage]

        similarities = []


        for i in range(len(sets)):

            for j in range(
                i + 1,
                len(sets)
            ):

                similarity = jaccard_similarity(
                    sets[i],
                    sets[j]
                )

                similarities.append(
                    similarity
                )


        if similarities:

            mean_stability = np.mean(
                similarities
            )

            min_stability = np.min(
                similarities
            )

        else:

            mean_stability = 1.0

            min_stability = 1.0


        percentage_label = int(
            percentage * 100
        )


        stability_metrics[
            f"top_{percentage_label}_mean_jaccard"
        ] = mean_stability


        stability_metrics[
            f"top_{percentage_label}_min_jaccard"
        ] = min_stability


    # --------------------------------------------------------
    # Score stability
    # --------------------------------------------------------

    scores_matrix = np.vstack(
        scores_by_seed
    )


    score_std_per_observation = np.std(
        scores_matrix,
        axis=0
    )


    mean_score_std = np.mean(
        score_std_per_observation
    )


    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    result = {

        "n_estimators": n_estimators,

        "max_samples": max_samples,

        "max_features": max_features,

        "mean_score_std": mean_score_std

    }


    result.update(
        stability_metrics
    )


    stability_results.append(
        result
    )


stability_results = pd.DataFrame(
    stability_results
)


# ============================================================
# STEP 3
# Merge Tuning and Stability Results
# ============================================================

evaluation_results = tuning_results.merge(
    stability_results,
    on=[
        "n_estimators",
        "max_samples",
        "max_features"
    ],
    how="left"
)


# ============================================================
# STEP 4
# Create Overall Stability Score
# ============================================================

evaluation_results[
    "stability_score"
] = (

    evaluation_results[
        "top_1_mean_jaccard"
    ]

    +

    evaluation_results[
        "top_5_mean_jaccard"
    ]

    +

    evaluation_results[
        "top_10_mean_jaccard"
    ]

) / 3


# ============================================================
# STEP 5
# Normalize Score Separation
# ============================================================

min_sep = evaluation_results[
    "score_separation"
].min()


max_sep = evaluation_results[
    "score_separation"
].max()


if max_sep > min_sep:

    evaluation_results[
        "normalized_separation"
    ] = (

        evaluation_results[
            "score_separation"
        ]
        - min_sep

    ) / (

        max_sep
        - min_sep

    )

else:

    evaluation_results[
        "normalized_separation"
    ] = 1.0


# ============================================================
# STEP 6
# Overall Tuning Score
# ============================================================

# Since the data are unlabeled, stability receives
# the highest weight.

evaluation_results[
    "overall_tuning_score"
] = (

    0.70
    *
    evaluation_results[
        "stability_score"
    ]

    +

    0.30
    *
    evaluation_results[
        "normalized_separation"
    ]

)


# ============================================================
# Rank Parameter Combinations
# ============================================================

evaluation_results = (

    evaluation_results

    .sort_values(
        "overall_tuning_score",
        ascending=False
    )

    .reset_index(
        drop=True
    )

)


evaluation_results[
    "parameter_rank"
] = np.arange(
    1,
    len(evaluation_results) + 1
)


# ============================================================
# Save Tuning Results
# ============================================================

evaluation_results.to_excel(
    tuning_output_file,
    index=False
)


# ============================================================
# STEP 7
# Select Best Hyperparameters
# ============================================================

best_parameters = evaluation_results.iloc[0]


BEST_N_ESTIMATORS = int(
    best_parameters[
        "n_estimators"
    ]
)


BEST_MAX_SAMPLES = best_parameters[
    "max_samples"
]


BEST_MAX_FEATURES = best_parameters[
    "max_features"
]


print("\n" + "=" * 70)
print("BEST HYPERPARAMETERS")
print("=" * 70)


print(
    f"n_estimators = "
    f"{BEST_N_ESTIMATORS}"
)


print(
    f"max_samples = "
    f"{BEST_MAX_SAMPLES}"
)


print(
    f"max_features = "
    f"{BEST_MAX_FEATURES}"
)


print(
    f"Overall tuning score = "
    f"{best_parameters['overall_tuning_score']:.4f}"
)


print(
    f"Stability score = "
    f"{best_parameters['stability_score']:.4f}"
)


print(
    f"Score separation = "
    f"{best_parameters['score_separation']:.6f}"
)


# ============================================================
# STEP 8
# Separate Contamination Analysis
# ============================================================

print("\n" + "=" * 70)
print("STEP 8: CONTAMINATION ANALYSIS")
print("=" * 70)


contamination_results = []


for contamination in CONTAMINATION_VALUES:

    print(
        f"Testing contamination = "
        f"{contamination}"
    )


    model = IsolationForest(
        n_estimators=BEST_N_ESTIMATORS,
        max_samples=BEST_MAX_SAMPLES,
        max_features=BEST_MAX_FEATURES,
        contamination=contamination,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


    model.fit(X)


    prediction = model.predict(X)


    anomaly_count = (
        prediction == -1
    ).sum()


    anomaly_percentage = (
        anomaly_count
        / len(X)
        * 100
    )


    contamination_results.append({

        "contamination": contamination,

        "anomaly_count": anomaly_count,

        "anomaly_percentage": anomaly_percentage

    })


contamination_results = pd.DataFrame(
    contamination_results
)


# ============================================================
# Save Stability and Contamination Results
# ============================================================

with pd.ExcelWriter(
    stability_output_file,
    engine="openpyxl"
) as writer:

    stability_results.to_excel(
        writer,
        sheet_name="Stability",
        index=False
    )


    contamination_results.to_excel(
        writer,
        sheet_name="Contamination",
        index=False
    )


# ============================================================
# STEP 9
# Final Isolation Forest Model
# ============================================================

print("\n" + "=" * 70)
print("STEP 9: FINAL ISOLATION FOREST")
print("=" * 70)


print(
    f"Final contamination = "
    f"{FINAL_CONTAMINATION}"
)


final_model = IsolationForest(

    n_estimators=BEST_N_ESTIMATORS,

    max_samples=BEST_MAX_SAMPLES,

    max_features=BEST_MAX_FEATURES,

    contamination=FINAL_CONTAMINATION,

    random_state=RANDOM_STATE,

    n_jobs=-1

)


# ============================================================
# Fit Final Model
# ============================================================

final_model.fit(X)


# ============================================================
# Calculate Final Anomaly Scores
# ============================================================

decision_function = (
    final_model.decision_function(X)
)


score_samples = (
    final_model.score_samples(X)
)


# Higher score = more anomalous

anomaly_score = -decision_function


# ============================================================
# Generate Final Predictions
# ============================================================

prediction = final_model.predict(X)


status = np.where(
    prediction == -1,
    "Anomaly",
    "Normal"
)


# ============================================================
# Create Final Results
# ============================================================

results = pd.DataFrame({

    "national_id": national_ids,

    "anomaly_score": anomaly_score,

    "decision_function": decision_function,

    "score_samples": score_samples,

    "prediction": prediction,

    "status": status

})


# ============================================================
# Rank Observations
# ============================================================

results = (

    results

    .sort_values(
        "anomaly_score",
        ascending=False
    )

    .reset_index(
        drop=True
    )

)


results["rank"] = np.arange(
    1,
    len(results) + 1
)


# ============================================================
# Organize Result Columns
# ============================================================

results = results[

    [

        "rank",

        "national_id",

        "anomaly_score",

        "decision_function",

        "score_samples",

        "prediction",

        "status"

    ]

]


# ============================================================
# Add Standardized Features
# ============================================================

results_with_features = results.merge(

    df[
        ["national_id"] + features
    ],

    on="national_id",

    how="left"

)


# ============================================================
# Add Risk Percentile
# ============================================================

results_with_features[
    "anomaly_percentile"
] = (

    results_with_features[
        "anomaly_score"
    ]

    .rank(
        pct=True
    )

    * 100

)


# ============================================================
# Save Final Results
# ============================================================

results_with_features.to_excel(

    final_output_file,

    index=False

)


# ============================================================
# Final Summary
# ============================================================

anomaly_count = (

    results["prediction"] == -1

).sum()


normal_count = (

    results["prediction"] == 1

).sum()


anomaly_percentage = (

    anomaly_count
    / len(results)
    * 100

)


print("\n" + "=" * 70)
print("FINAL MODEL SUMMARY")
print("=" * 70)


print(
    f"Total observations: "
    f"{len(results):,}"
)


print(
    f"Anomalies: "
    f"{anomaly_count:,}"
)


print(
    f"Normal: "
    f"{normal_count:,}"
)


print(
    f"Anomaly rate: "
    f"{anomaly_percentage:.2f}%"
)


print("\nFinal parameters:")


print(
    f"n_estimators = "
    f"{BEST_N_ESTIMATORS}"
)


print(
    f"max_samples = "
    f"{BEST_MAX_SAMPLES}"
)


print(
    f"max_features = "
    f"{BEST_MAX_FEATURES}"
)


print(
    f"contamination = "
    f"{FINAL_CONTAMINATION}"
)


print("\nOutput files:")


print(
    f"1. {tuning_output_file}"
)


print(
    f"2. {stability_output_file}"
)


print(
    f"3. {final_output_file}"
)


print("\nAnalysis completed successfully.")


STEP 1: HYPERPARAMETER SENSITIVITY ANALYSIS
Testing combination 1/36: n_estimators=100, max_samples=0.5, max_features=0.5
Testing combination 2/36: n_estimators=100, max_samples=0.5, max_features=0.75
Testing combination 3/36: n_estimators=100, max_samples=0.5, max_features=1.0
Testing combination 4/36: n_estimators=100, max_samples=0.75, max_features=0.5
Testing combination 5/36: n_estimators=100, max_samples=0.75, max_features=0.75
Testing combination 6/36: n_estimators=100, max_samples=0.75, max_features=1.0
Testing combination 7/36: n_estimators=100, max_samples=1.0, max_features=0.5
Testing combination 8/36: n_estimators=100, max_samples=1.0, max_features=0.75
Testing combination 9/36: n_estimators=100, max_samples=1.0, max_features=1.0
Testing combination 10/36: n_estimators=200, max_samples=0.5, max_features=0.5
Testing combination 11/36: n_estimators=200, max_samples=0.5, max_features=0.75
Testing combination 12/36: n_estimators=200, max_samples=0.5, max_features=1.0
Testing c

c:\Users\NITRO\anaconda3\envs\environment\lib\site-packages\sklearn\base.py:451: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  "X does not have valid feature names, but"


Testing contamination = 0.02


c:\Users\NITRO\anaconda3\envs\environment\lib\site-packages\sklearn\base.py:451: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  "X does not have valid feature names, but"


Testing contamination = 0.05


c:\Users\NITRO\anaconda3\envs\environment\lib\site-packages\sklearn\base.py:451: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  "X does not have valid feature names, but"


Testing contamination = 0.1


c:\Users\NITRO\anaconda3\envs\environment\lib\site-packages\sklearn\base.py:451: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  "X does not have valid feature names, but"



STEP 9: FINAL ISOLATION FOREST
Final contamination = 0.05


c:\Users\NITRO\anaconda3\envs\environment\lib\site-packages\sklearn\base.py:451: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  "X does not have valid feature names, but"



FINAL MODEL SUMMARY
Total observations: 3,008
Anomalies: 151
Normal: 2,857
Anomaly rate: 5.02%

Final parameters:
n_estimators = 500
max_samples = 0.75
max_features = 1.0
contamination = 0.05

Output files:
1. isolation_forest_tuning.xlsx
2. isolation_forest_stability.xlsx
3. isolation_forest_final_results.xlsx

Analysis completed successfully.
